# 04 — Sepsis Supplementary Analysis

Full learner grid, oracle-selected estimator results, ranking metrics,
random partition analysis, and detailed support/overlap tables.

These results belong in the supplementary material of the paper.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'sepsis'))
import config as CFG

from helpers.runner import load_checkpoint
from helpers.metrics import (
    build_full_learner_table, all_metrics, ranking_metrics_within_groups, ranking_comparison_table,
    resolve_alternative, describe_paired_test,
    METHODS_ORDER, LEARNERS_ORDER
)
from helpers.plotting import METHOD_LABELS

results_by_seed = load_checkpoint(CFG.CHECKPOINT_PATH)
completed_seeds = sorted(results_by_seed.keys())
print(f'Loaded {len(completed_seeds)} seeds.')


## 1. Full Learner Grid — ATE MSE

In [ ]:
ate_grid = build_full_learner_table(results_by_seed, metric='ate_mse')
print('ATE MSE: mean across seeds (rows=learners, cols=methods)')
display(ate_grid.round(5))
ate_grid.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_ate_mse.csv'))

In [ ]:
baseline_methods = [m for m in METHODS_ORDER if m != 'CDV_SEPARATE']
cdv_ate_row = ate_grid.loc['CDV_SEPARATE']
ate_improvement = (ate_grid.loc[baseline_methods] - cdv_ate_row) / ate_grid.loc[baseline_methods] * 100
ate_improvement.index = [METHOD_LABELS.get(m, m) for m in ate_improvement.index]
print('CDV_SEPARATE improvement (%) over each method, per learner (ATE MSE):')
display(ate_improvement.round(2))
ate_improvement.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_ate_mse_improvement.csv'))

## 2. Full Learner Grid — CATE MSE

In [ ]:
cate_grid = build_full_learner_table(results_by_seed, metric='cate_mse')
print('CATE MSE: mean across seeds')
display(cate_grid.round(5))
cate_grid.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_cate_mse.csv'))

In [ ]:
cdv_cate_row = cate_grid.loc['CDV_SEPARATE']
cate_improvement = (cate_grid.loc[baseline_methods] - cdv_cate_row) / cate_grid.loc[baseline_methods] * 100
cate_improvement.index = [METHOD_LABELS.get(m, m) for m in cate_improvement.index]
print('CDV_SEPARATE improvement (%) over each method, per learner (CATE MSE):')
display(cate_improvement.round(2))
cate_improvement.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_cate_mse_improvement.csv'))

## 3. Ranking Metrics — Kendall τ and Spearman ρ (DR-RF)

In [ ]:
print(f'Ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds')
ranking_rows = []
for method in METHODS_ORDER:
    tau_vals, rho_vals = [], []
    for sr in results_by_seed.values():
        m = sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {})
        tau_vals.append(m.get('kendall_tau', np.nan))
        rho_vals.append(m.get('spearman_rho', np.nan))
    ranking_rows.append({
        'Method': METHOD_LABELS.get(method, method),
        'Kendall τ mean': np.nanmean(tau_vals),
        'Kendall τ std':  np.nanstd(tau_vals),
        'Spearman ρ mean': np.nanmean(rho_vals),
        'Spearman ρ std':  np.nanstd(rho_vals),
        'N seeds': int(np.sum(np.isfinite(tau_vals))),
    })
ranking_df = pd.DataFrame(ranking_rows)
display(ranking_df.round(4))
ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics.csv'), index=False)

In [ ]:
selected_learner_rank = CFG.PRIMARY_LEARNER  # change to compare a different learner

print(f'CDV_SEPARATE vs other methods — Kendall τ ({selected_learner_rank})')
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
tau_cmp = ranking_comparison_table(results_by_seed, selected_learner_rank, metric='kendall_tau',
                                    ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
tau_cmp.index = [METHOD_LABELS.get(m, m) for m in tau_cmp.index]

# Add CDV row at top
cdv_tau_vals = np.array([
    sr.get("metrics", {}).get("CDV_SEPARATE", {}).get(selected_learner_rank, {}).get('kendall_tau', np.nan)
    for sr in results_by_seed.values()
])
n_seeds_tau = int(np.sum(np.isfinite(cdv_tau_vals)))
cdv_tau_row = pd.DataFrame([{
    'method': 'CDV_SEPARATE',
    'mean': float(np.nanmean(cdv_tau_vals)),
    'std': float(np.nanstd(cdv_tau_vals)),
    'paired 95% CI of Δ (CDV_SEPARATE − method)': '-',
    'p_value (paired)': '-',
    'N seeds': n_seeds_tau
}])
tau_cmp = tau_cmp.reset_index().rename(columns={'index': 'method'})
tau_cmp = pd.concat([cdv_tau_row, tau_cmp], ignore_index=True).set_index('method')
display(tau_cmp.round(4))

print(f'\nCDV_SEPARATE vs other methods — Spearman ρ ({selected_learner_rank})')
print(describe_paired_test('CDV_SEPARATE', 'method', 'Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
rho_cmp = ranking_comparison_table(results_by_seed, selected_learner_rank, metric='spearman_rho',
                                    ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
rho_cmp.index = [METHOD_LABELS.get(m, m) for m in rho_cmp.index]

# Add CDV row at top
cdv_rho_vals = np.array([
    sr.get("metrics", {}).get("CDV_SEPARATE", {}).get(selected_learner_rank, {}).get('spearman_rho', np.nan)
    for sr in results_by_seed.values()
])
n_seeds_rho = int(np.sum(np.isfinite(cdv_rho_vals)))
cdv_rho_row = pd.DataFrame([{
    'method': 'CDV_SEPARATE',
    'mean': float(np.nanmean(cdv_rho_vals)),
    'std': float(np.nanstd(cdv_rho_vals)),
    'paired 95% CI of Δ (CDV_SEPARATE − method)': '-',
    'p_value (paired)': '-',
    'improvement_pct': np.nan,
}])
rho_cmp = rho_cmp.reset_index().rename(columns={'index': 'method'})
rho_cmp = pd.concat([cdv_rho_row, rho_cmp], ignore_index=True).set_index('method')
display(rho_cmp.round(4))

tau_cmp.to_csv(os.path.join(CFG.ARTIFACTS_DIR, f'ranking_metrics_tau_{selected_learner_rank}.csv'), index=False)
rho_cmp.to_csv(os.path.join(CFG.ARTIFACTS_DIR, f'ranking_metrics_rho_{selected_learner_rank}.csv'), index=False)


## 4. Within-CDV Ranking Metrics — Kendall τ and Spearman ρ (primary learner)

Section 3's metrics pool predictions across all CDV groups (and OTHER) before ranking.
Here, Kendall τ / Spearman ρ are computed separately WITHIN each CDV group, then
size-weighted averaged across groups — isolating within-subgroup ranking quality from
across-group ordering effects.


In [ ]:
print(f'Within-CDV ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds')
within_rows = []
for method in METHODS_ORDER:
    tau_vals, rho_vals = [], []
    for sr in results_by_seed.values():
        pred = sr.get('predictions', {}).get(method, {}).get(CFG.PRIMARY_LEARNER, {})
        ite_pred = np.asarray(pred.get('ite_pred', []))
        ite_true = np.asarray(pred.get('ite_true', []))
        variant = np.asarray(pred.get('variant', []))
        if len(ite_pred) == 0:
            continue
        m = ranking_metrics_within_groups(ite_pred, ite_true, variant)
        tau_vals.append(m['kendall_tau_within'])
        rho_vals.append(m['spearman_rho_within'])
    within_rows.append({
        'Method': METHOD_LABELS.get(method, method),
        'Kendall τ (within-CDV) mean': np.nanmean(tau_vals),
        'Kendall τ (within-CDV) std':  np.nanstd(tau_vals),
        'Spearman ρ (within-CDV) mean': np.nanmean(rho_vals),
        'Spearman ρ (within-CDV) std':  np.nanstd(rho_vals),
        'N seeds': int(np.sum(np.isfinite(tau_vals))),
    })

within_ranking_df = pd.DataFrame(within_rows)
display(within_ranking_df.round(4))
within_ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics_within_cdv.csv'), index=False)


## 5. Oracle-Selected Estimator Results

In [ ]:
oracle_rows = []
for seed, sr in results_by_seed.items():
    for method in METHODS_ORDER:
        oracle = sr.get('oracle', {}).get(method, {})
        m = oracle.get('metrics', {})
        oracle_rows.append({
            'outer_seed':       seed,
            'method':           method,
            'selected_learner': oracle.get('selected_learner', 'N/A'),
            'ate_mse':          m.get('ate_mse', np.nan),
            'cate_mse':         m.get('cate_mse', np.nan),
            'kendall_tau':      m.get('kendall_tau', np.nan),
            'spearman_rho':     m.get('spearman_rho', np.nan),
        })

oracle_df = pd.DataFrame(oracle_rows)
print('Oracle results by method:')
display(oracle_df.groupby('method')[['ate_mse', 'cate_mse']].mean().round(5))

print('\nLearner selection frequency:')
display(oracle_df.groupby(['method', 'selected_learner']).size().unstack(fill_value=0))

oracle_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'oracle_results.parquet'), index=False)
print(f'Saved: oracle_results.parquet')

## 6. Random Partition Analysis

In [ ]:
# Show that MATCHED_RANDOM_PARTITIONS results are based on averaged permutations
rp_rows = []
for seed, sr in results_by_seed.items():
    rp_preds = sr.get('predictions', {}).get('MATCHED_RANDOM_PARTITIONS', {})
    for lname, pred in rp_preds.items():
        n_perms = len(pred.get('perm_ite_preds', []))
        rp_rows.append({'outer_seed': seed, 'learner': lname, 'n_permutations': n_perms})

if rp_rows:
    rp_df = pd.DataFrame(rp_rows)
    print('Permutations per seed × learner (should all be N_RANDOM_PERMUTATIONS):')
    display(rp_df.groupby('learner')['n_permutations'].describe())
else:
    print('No random partition data found.')

## 7. Build Tidy Main Results DataFrame

In [ ]:
main_rows = []
for seed, sr in results_by_seed.items():
    for method in METHODS_ORDER:
        for learner in LEARNERS_ORDER:
            m = sr.get('metrics', {}).get(method, {}).get(learner, {})
            main_rows.append({
                'dataset':           'sepsis',
                'outer_seed':        seed,
                'alpha':             np.nan,
                'method':            method,
                'learner':           learner,
                'ate_mse':           m.get('ate_mse', np.nan),
                'cate_mse':          m.get('cate_mse', np.nan),
                'kendall_tau':       m.get('kendall_tau', np.nan),
                'spearman_rho':      m.get('spearman_rho', np.nan),
                'n_train':           sr.get('n_train', np.nan),
                'n_test':            sr.get('n_test', np.nan),
                'n_retained_cdvs':   len(sr.get('retained_cdv_info', {})),
                'pct_other_train':   sr.get('pct_other_train', np.nan),
                'pct_other_test':    sr.get('pct_other_test', np.nan),
            })

main_df = pd.DataFrame(main_rows)
main_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'main_results.parquet'), index=False)
print(f'Tidy results DataFrame: {main_df.shape}')
print(f'Saved: {CFG.ARTIFACTS_DIR}/main_results.parquet')
display(main_df[main_df['learner'] == 'DR_RF'].groupby('method')[['ate_mse', 'cate_mse']].mean().round(5))